# 🎭 AI Companion: Universal Roleplay Bridge (Threaded)

This notebook acts as a remote 'brain' for your AI Companion. It allows you to run high-end models on Google's T4 GPUs or Kaggle and tunnel the response back to your local machine via Cloudflare Quick Tunnels.

### 🛠️ Setup Instructions:
1. **GPU Acceleration**: Go to `Runtime` > `Change runtime type` and ensure **T4 GPU** (or **T4 x2** on Kaggle) is selected.
2. **Secrets (IMPORTANT)**: 
   - **Colab**: Click the **Key icon** (Secrets) on the left sidebar.
   - **Kaggle**: Go to **Add-ons** > **Secrets**.
   - Add `HF_TOKEN` with your [HuggingFace Token](https://huggingface.co/settings/tokens).
   - No token required for Cloudflare Quick Tunnels! They are created on-demand.
   - Enable access for the notebook.
3. **Run All**: Press `Ctrl + F9` or go to `Runtime` > `Run all`.

### 🔗 Connecting to the Local App:
1. Wait for the final cell to display the **🚀 BRIDGE ONLINE!** message.
2. Copy the **URL** (it will look like `https://xxxx.trycloudflare.com`).
3. Open your local `settings.json` and paste the URL into `remote_llm_url`.
4. Restart your local `main.py` script.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Simply change this variable to "colab" or "kaggle" based on your environment.
ENV_TYPE = "kaggle"

# VS Code Colab Runtime Fix
if not os.environ.get("HF_TOKEN"):
    for path in ["/content/.env", os.path.expanduser("~/.env")]:
        if os.path.exists(path):
            load_dotenv(path)
            break

def get_secret(name):
    val = os.environ.get(name)
    if val: return val.strip()
    if ENV_TYPE == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name).strip()
        except: pass
    elif ENV_TYPE == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name).strip()
        except: pass
    return None

print(f"Detected Environment: {ENV_TYPE}")

In [ ]:
# @title 1. Install Dependencies
!pip install -q -U fastapi uvicorn pyngrok nest_asyncio requests python-dotenv
!pip install -q -U transformers accelerate
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U bitsandbytes nvidia-cuda-runtime-cu12 nvidia-nvjitlink-cu12

In [ ]:
# @title 2. Load Dual-Worker Specialist Pool
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
HF_TOKEN = get_secret('HF_TOKEN')

model_id = "Sao10K/L3-8B-Stheno-v3.2"

print(f"Loading Dual-Worker Pool for {model_id}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Worker 0 - GPU 0
print("🚀 Loading Worker 0 on GPU 0...")
tokenizer_0 = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
model_0 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True,
    token=HF_TOKEN
)

# Worker 1 - GPU 1 (If available)
model_1, tokenizer_1 = None, None
if torch.cuda.device_count() > 1:
    print("🚀 Loading Worker 1 on GPU 1...")
    tokenizer_1 = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    model_1 = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="cuda:1",
        trust_remote_code=True,
        token=HF_TOKEN
    )
else:
    print("⚠️ Only one GPU detected. Dual-Worker pool will run in single-mode.")

print("\n🚀 GPU Allocation Breakdown:")
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    used = torch.cuda.memory_allocated(i) / 1024**3
    print(f"  [GPU {i}]: {used:.2f}GB / {mem:.2f}GB used")

print(f"\n✅ Dual-Worker Pool READY!")

In [ ]:
# @title 3. Start Dual-Worker API Server
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import uvicorn, nest_asyncio, re, os, time, threading, queue
# Cloudflare Quick Tunnel - no dependencies needed
from pydantic import BaseModel
from typing import List
from threading import Thread
from transformers import TextIteratorStreamer
import torch

# No NGROK_TOKEN required for Cloudflare Quick Tunnels

app = FastAPI()
nest_asyncio.apply()

# Per-GPU locks to ensure one-task-per-GPU
gpu0_lock = threading.Lock()
gpu1_lock = threading.Lock()

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    max_tokens: int = 1024
    temperature: float = 0.8
    n: int = 1

def generate_worker_task(worker_id, model_obj, tokenizer_obj, lock, chat, max_tokens, temperature, out_list):
    with lock:
        print(f"  [Worker {worker_id}] Generating candidate...")
        inputs = tokenizer_obj.apply_chat_template(
            chat, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model_obj.device)
        
        try:
            with torch.no_grad():
                output_tokens = model_obj.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature if temperature > 0 else 1.0,
                    do_sample=True if temperature > 0 else False,
                    num_return_sequences=1,
                    pad_token_id=tokenizer_obj.eos_token_id,
                    use_cache=True
                )
            input_len = inputs.input_ids.shape[1]
            out_list.append(tokenizer_obj.decode(output_tokens[0][input_len:], skip_special_tokens=True).strip())
        finally:
            torch.cuda.empty_cache()

@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    chat = [{"role": m.role, "content": m.content} for m in request.messages]
    
    if request.n > 1:
        print(f"🎭 [POOL] Dispatching {request.n} candidates across workers...")
        results = []
        threads = []
        
        task_queue = queue.Queue()
        for _ in range(request.n): task_queue.put(True)
        
        def pool_manager(worker_id, model_obj, tokenizer_obj, lock):
            while True:
                try:
                    _ = task_queue.get_nowait()
                    generate_worker_task(worker_id, model_obj, tokenizer_obj, lock, chat, request.max_tokens, request.temperature, results)
                    task_queue.task_done()
                except queue.Empty:
                    break
                except Exception as e:
                    print(f"❌ Worker {worker_id} Error: {e}")
                    break

        # Start management threads
        t0 = Thread(target=pool_manager, args=(0, model_0, tokenizer_0, gpu0_lock))
        t0.start()
        threads.append(t0)
        
        if model_1:
            t1 = Thread(target=pool_manager, args=(1, model_1, tokenizer_1, gpu1_lock))
            t1.start()
            threads.append(t1)
            
        for t in threads: t.join()
        print(f"✅ [POOL] Success! Generated {len(results)} candidates.")
        return {"candidates": results}

    else:
        # Stream Router: Grab first available GPU
        worker_id = 0
        if model_1 and gpu0_lock.locked() and not gpu1_lock.locked():
            worker_id = 1
        
        target_lock = gpu1_lock if worker_id == 1 else gpu0_lock
        target_model = model_1 if worker_id == 1 else model_0
        target_tokenizer = tokenizer_1 if worker_id == 1 else tokenizer_0
        
        print(f"💬 [STREAM] Worker {worker_id} Thinking... (History: {len(chat)})")
        
        def stream_with_lock():
            with target_lock:
                inputs = target_tokenizer.apply_chat_template(
                    chat, add_generation_prompt=True, return_tensors="pt", return_dict=True
                ).to(target_model.device)
                
                streamer = TextIteratorStreamer(target_tokenizer, skip_prompt=True, skip_special_tokens=True)
                generation_kwargs = {
                    **inputs, "streamer": streamer, "max_new_tokens": request.max_tokens,
                    "temperature": request.temperature if request.temperature > 0 else 1.0,
                    "do_sample": True if request.temperature > 0 else False,
                    "use_cache": True
                }
                t = Thread(target=target_model.generate, kwargs=generation_kwargs)
                t.start()
                
                try:
                    for text in streamer: yield text
                finally:
                    torch.cuda.empty_cache()
                    print(f"✅ [STREAM] Worker {worker_id} Complete. Cache Cleared.")
        
        return StreamingResponse(stream_with_lock(), media_type="text/plain")

# Cloudflare tunnel will be started below
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

if server_thread.is_alive():
    import subprocess
    import time
    
    # Download cloudflared if not present
    import os
    cf_path = os.path.expanduser("~/.cloudflared/cloudflared")
    if not os.path.exists(cf_path):
        os.makedirs(os.path.dirname(cf_path), exist_ok=True)
        print("Downloading cloudflared...")
        !curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {cf_path}
        !chmod +x {cf_path}
    
    # Start cloudflared tunnel
    print("Starting Cloudflare Quick Tunnel...")
    proc = subprocess.Popen(
        [cf_path, "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Extract URL from stderr (cloudflared outputs here)
    public_url = None
    try:
        for _ in range(15):  # Try for ~15 seconds
            if proc.poll() is not None:
                # Process ended
                break
            # Read from stderr (non-blocking)
            line = proc.stderr.readline() if proc.stderr else ""
            if line:
                print(f"[cloudflared] {line.strip()}")
                if '.trycloudflare.com' in line:
                    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
                    if match:
                        public_url = match.group(0)
                        break
            time.sleep(1)
    except Exception as e:
        print(f"Error reading tunnel output: {e}")
    
    if public_url:
        print(f"\n🚀 BRIDGE ONLINE!\nURL: {public_url}\n")
    else:
        print("❌ Failed to extract Cloudflare URL")
else:
    print("❌ SERVER ERROR")

try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    print("Bridge stopped.")